Загрузка датасета

In [1]:
import pandas as pd

df = pd.read_excel('../data/Onlineretail.xlsx')

Обзор датасета

In [ ]:
print(df.head())
df.info()
print(df.describe())
df.isna().sum()

Переименование столбцов

In [3]:
df.columns = df.columns.str.lower()

Поиск отрицательных и равных нулю значений

In [ ]:
print(df[df['quantity'] <= 0])
print(df[df['unitprice'] < 0])
print(df[df['unitprice'] == 0])
print(df[df['unitprice'] > 30000])

Выделяющиеся коды товара и работа с ними

In [ ]:
print(df['stockcode'][df['stockcode'].str.len() > 8].unique())
print(df['stockcode'][df['stockcode'].str.len() < 5].unique())

In [ ]:
print(df[df['stockcode'] == 'POST'])
print(df[df['stockcode'] == 'D'])
print(df[df['stockcode'] == 'C2'])
print(df[df['stockcode'] == 'DOT'])
print(df[df['stockcode'] == 'M'])
print(df[df['stockcode'] == 'S'])
print(df[df['stockcode'] == 'm'])
print(df[df['stockcode'] == 'PADS'])
print(df[df['stockcode'] == 'B'])
print(df[df['stockcode'] == 'CRUK'])
print(df[df['stockcode'] == 'BANK CHARGES'].head())
print(df[df['stockcode'] == 'AMAZONFEE'].head())
print(df[df['stockcode'].str.contains('gift', na=False)].head())

Добавление необходимых для анализа столбцов и очистка от не товаров

In [108]:
df['revenue'] = df['quantity'] * df['unitprice']
df['month'] = df['invoicedate'].dt.month_name()
df2 = df.copy()
df2 = df2[~df2['stockcode'].isin(['POST', 'D', 'C2', 'DOT', 'M', 'S', 'm', 'PADS', 'B', 'CRUK',
                                  'BANK CHARGES', 'AMAZONFEE', 'gift_0001_40',
                                  'gift_0001_50', 'gift_0001_30', 'gift_0001_20'])]

Работа со столбцом 'description'

In [ ]:
descriptions = (df2.groupby('description', dropna=False).agg(
    rows = ('description', 'size'),
    total_revenue = ('revenue', 'sum'),
    min_quantity = ('quantity', 'min'),
    max_quantity = ('quantity', 'max'),
    avg_quantity = ('quantity', 'mean'),
    min_price = ('unitprice', 'min'),
    max_price = ('unitprice', 'max'),
    avg_price = ('unitprice', 'mean')
)
).sort_values('rows', ascending=False).reset_index()

print(descriptions['description'][(descriptions['rows'] < 20) & \
                                  (descriptions['description'].str.len() < 18) & \
                                    ((descriptions['total_revenue'] > 200) | \
                                     (descriptions['total_revenue'] < -200))].unique(), '\n')
print(descriptions['description'][(descriptions['rows'] < 20) & \
                                  (descriptions['description'].str.len() < 18)].unique())

Ключевой анализ

In [110]:
best_month = (df.groupby('month').agg({'revenue': 'sum', 'quantity': 'sum'})
                                      .reset_index().sort_values('revenue', ascending=False))
print(best_month, '\n')
print(best_month['revenue'].sum())

        month      revenue  quantity
9    November  1461756.250    740286
2    December  1182625.030    568561
10    October  1070704.670    570532
11  September  1019687.622    549817
8         May   723333.510    380391
6        June   691123.120    341623
7       March   683267.080    351872
1      August   682680.510    406199
5        July   681300.111    391116
4     January   560000.260    308966
3    February   498062.650    277989
0       April   493207.121    289098 

9747747.934


In [115]:
countries_revenue = df.groupby('country')['revenue'].sum().reset_index()\
    .sort_values('revenue', ascending=False).reset_index()
countries_revenue.drop(columns=['index'], inplace=True)
print(countries_revenue.head(10))
print(countries_revenue)

          country      revenue
0  United Kingdom  8187806.364
1     Netherlands   284661.540
2            EIRE   263276.820
3         Germany   221698.210
4          France   197403.900
5       Australia   137077.270
6     Switzerland    56385.350
7           Spain    54774.580
8         Belgium    40910.960
9          Sweden    36595.910
                 country      revenue
0         United Kingdom  8187806.364
1            Netherlands   284661.540
2                   EIRE   263276.820
3                Germany   221698.210
4                 France   197403.900
5              Australia   137077.270
6            Switzerland    56385.350
7                  Spain    54774.580
8                Belgium    40910.960
9                 Sweden    36595.910
10                 Japan    35340.620
11                Norway    35163.460
12              Portugal    29367.020
13               Finland    22326.740
14       Channel Islands    20086.290
15               Denmark    18768.140
16           

In [120]:
top_prods = df2.groupby('description')['revenue'].sum().reset_index()\
    .sort_values('revenue', ascending=False).reset_index()
top_prods.drop(columns=['index'], inplace=True)
print(top_prods.head(10))
print(top_prods[top_prods['revenue'] > 0].tail(10))

                          description    revenue
0            REGENCY CAKESTAND 3 TIER  164762.19
1  WHITE HANGING HEART T-LIGHT HOLDER   99668.47
2                       PARTY BUNTING   98302.98
3             JUMBO BAG RED RETROSPOT   92356.03
4                  RABBIT NIGHT LIGHT   66756.59
5     PAPER CHAIN KIT 50'S CHRISTMAS    63791.94
6       ASSORTED COLOUR BIRD ORNAMENT   58959.73
7                       CHILLI LIGHTS   53768.06
8                      SPOTTY BUNTING   42065.32
9             JUMBO BAG PINK POLKADOT   41619.66
                              description  revenue
3994           WIRE FLOWER T-LIGHT HOLDER     1.25
3995  60 GOLD AND SILVER FAIRY CAKE CASES     1.10
3996       3 WICK CHRISTMAS BRIAR CANDLE      0.97
3997       HAPPY BIRTHDAY CARD TEDDY/CAKE     0.95
3998       CAT WITH SUNGLASSES BLANK CARD     0.95
3999      PINK CRYSTAL GUITAR PHONE CHARM     0.85
4000             VINTAGE BLUE TINSEL REEL     0.84
4001  WHITE BEADED GARLAND STRING 20LIGHT     0.80
40